In [ ]:
"""
Step 03: Timestamp Segment (TSSEG) Data Collection

Objective:

    - One row per minute: cum_max (running maximum high since the start of the data), minima / maxima counts and the
      distance (price, percent, bars) from the latest confirmed swing (the segment anchor).
    - A new segment starts when a swing is confirmed (3 bars after the swing bar), so seg_cs_count resets to 3.
    - Logic unchanged from the step08_TSSEG_data_collection notebook of the previous workspace (snapshot_features.get_date_TSSEG_pdf).
      The cumulative maximum is computed in memory instead of being saved per year.
      Existing TSSEG_data_YYYYMMDD.csv files can be copied into step03_TSSEG_data instead of being regenerated.

Prerequisite: step01.
"""
None

In [ ]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
# IMPORT PLOTLY
import plotly.graph_objects as go
from plotly.subplots import make_subplots
# IMPORT THE SHARED CONFIGURATION
from so import config
# IMPORT TRANSFORM OHLCV DATA FUNCTIONS
from so.features.ohlcv_data_utils import format_ohlcv_pdf
# IMPORT SNAPSHOT AND CONTEXT FEATURE FUNCTIONS
from so.features.snapshot_features import get_date_TSSEG_pdf
from so.features.context_features import add_cum_max_col
# IMPORT PLOT OHLCV FEATURES FUNCTIONS
from so.features.plot_ohlcv_utils import fig_add_ohlcv
# IMPORT MARKET DATETIME FUNCTIONS
from so.core.datetime_utils import ny_tz
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import get_path_file_list, \
                                    read_pdf_from_csv_file_path_list, \
                                    read_date_range_csv_files_from_path, \
                                    generate_func_data_date_list, \
                                    get_missing_date_list
# IMPORT PATHS
from so import paths

In [ ]:
# CALL FUNCTION TO GET ALL FILES IN PATH
rawzone_data_file_path_list = get_path_file_list(paths.LOCAL_OHLCV_DATA_FILE_PATH_STR)[:]
# CALL FUNCTION TO READ PDF FROM PATH LIST
raw_ohlcv_pdf = read_pdf_from_csv_file_path_list(rawzone_data_file_path_list)
# CORRECT THE TIMESTAMP DATE COLUMN
raw_ohlcv_pdf["timestamp"] = pd.to_datetime(raw_ohlcv_pdf.timestamp, utc=True).dt.tz_convert(ny_tz)
raw_ohlcv_pdf["date"] = raw_ohlcv_pdf["timestamp"].dt.date
# CALL FUNCTION TO COLLECT DATA AND SORT BY TIMESTAMP
complete_ohlcv_pdf = format_ohlcv_pdf(raw_ohlcv_pdf.copy()).sort_values("timestamp").reset_index(drop=True)
# DISPLAY INFORMATION
print(f"min timestamp:\t{complete_ohlcv_pdf.timestamp.min()}")
print(f"max timestamp:\t{complete_ohlcv_pdf.timestamp.max()}")
# PREVIEW DATAFRAME
complete_ohlcv_pdf

In [ ]:
# CALL FUNCTION TO ADD THE CUMULATIVE MAXIMUM HIGH (RUNNING MAXIMUM SINCE THE FIRST ROW OF THE DATA)
complete_ohlcv_pdf = add_cum_max_col(complete_ohlcv_pdf)
# CREATE THE TIMESTAMP TO CUMULATIVE MAXIMUM DICTIONARY
ts_cum_max_dict = dict(zip(complete_ohlcv_pdf["timestamp"], complete_ohlcv_pdf["cum_max"]))

# FUNCTION: GENERATE THE DATE TIMESTAMP SEGMENT (TSSEG) DATAFRAME
def get_date_TSSEG_data_pdf(date_str_in):
    # CALL FUNCTION TO GENERATE THE DATE TSSEG DATAFRAME (READS OR CREATES THE STEP01 SNAPSHOTS)
    return get_date_TSSEG_pdf(complete_ohlcv_pdf, ts_cum_max_dict, date_str_in)

# DEFINE SAMPLE DATE
sample_date_str = str(complete_ohlcv_pdf["date"].iloc[-1])
# CALL FUNCTION TO GENERATE THE SAMPLE DATE TSSEG DATAFRAME
sample_date_TSSEG_pdf = get_date_TSSEG_data_pdf(sample_date_str)
# PREVIEW DATAFRAME
sample_date_TSSEG_pdf

In [ ]:
# FUNCTION: PLOT OHLCV + PREVIOUS SEGMENT DELTA AS SUBPLOTS
def plot_ohlcv_prev_seg_delta(pdf_in, width_in=1100, height_in=700):
    # CREATE SUBPLOTS: 1 COLUMN, 2 ROWS, SHARED X-AXIS
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05, subplot_titles=("OHLCV", "Prev Seg Delta Pct"))
    # PLOT CANDLESTICKS
    fig_add_ohlcv(fig, pdf_in, row_in=1, col_in=1)
    # PLOT PREV SEGMENT DELTA
    fig.add_trace(go.Scatter(x=pdf_in["timestamp"], y=pdf_in["prev_seg_delta_pct"], mode="markers+lines",
                             marker=dict(color="yellow", size=5), line=dict(color="cyan", width=2), name="Prev Seg Delta Pct"), row=2, col=1)
    # ADD ZERO REFERENCE LINE
    fig.add_hline(y=0, line_color="white", line_width=1, row=2, col=1)
    # UPDATE LAYOUT (DARK MODE)
    fig.update_layout(template="plotly_dark", width=width_in, height=height_in, showlegend=True, xaxis_rangeslider_visible=False,
                      font=dict(color="white"), title="Candlestick + Prev Seg Delta Comparison")
    # SHOW FIGURE
    fig.show()

# CALL FUNCTION TO PLOT CANDLESTICK AND DELTA
plot_ohlcv_prev_seg_delta(sample_date_TSSEG_pdf)

In [ ]:
# COLLECT THE START AND END DATE
start_date_str, end_date_str = str(complete_ohlcv_pdf["date"].min()), str(complete_ohlcv_pdf["date"].max())
# GET MISSING DATE LIST
missing_date_list = get_missing_date_list(paths.LOCAL_TSSEG_DATA_FILE_PATH_STR, start_date_str, end_date_str)
# PREVIEW LIST
missing_date_list[:10]

In [ ]:
# CALL FUNCTION TO PROCESS DATE LIST
generate_func_data_date_list(get_date_TSSEG_data_pdf, missing_date_list, paths.LOCAL_TSSEG_DATA_FILE_PATH_STR, "TSSEG_data", "I")

In [ ]:
# CALL FUNCTION TO READ THE LAST MONTH OF TSSEG DATA
TSSEG_data_pdf = read_date_range_csv_files_from_path(paths.LOCAL_TSSEG_DATA_FILE_PATH_STR, str((pd.to_datetime(end_date_str) - pd.DateOffset(months=1)).date()), end_date_str)
# PREVIEW DATAFRAME
TSSEG_data_pdf

In [ ]:
"""
Verification: the TSSEG files were COPIED from the previous workspace (old step08), not regenerated. This cell regenerates 5
random dates with the current code and compares them with the stored files (CSV round trip, string compare).
Not run yet (2026-10-04): run this cell once and record the result in docs/RESULTS_LOG.md.
"""
# IMPORT THE HELPERS
import io, random
# PICK 5 RANDOM DATES (FIXED SEED)
check_date_list = random.Random(0).sample(sorted(complete_ohlcv_pdf["date"].unique()), 5)
# ITERATE OVER THE DATES
for check_date in check_date_list:
    # REGENERATE THE DATE WITH THE CURRENT CODE (CSV ROUND TRIP)
    new_pdf = pd.read_csv(io.StringIO(get_date_TSSEG_data_pdf(str(check_date)).to_csv(index=False)))
    # READ THE STORED FILE
    old_pdf = pd.read_csv(f"{paths.LOCAL_TSSEG_DATA_FILE_PATH_STR}TSSEG_data_{check_date:%Y%m%d}.csv").drop(columns=["date"], errors="ignore")
    # COMPARE THE COLUMNS AND THE VALUES
    same_bool = list(old_pdf.columns) == list(new_pdf.columns) and old_pdf.astype(str).equals(new_pdf.astype(str))
    # DISPLAY INFORMATION
    print(f"{check_date}\t{'✅ identical' if same_bool else '❌ differs'}")